# Guardrails

*Notebook 22*

Add input and output validation to keep your agents safe, on-topic, and production-ready.
<br>
<br>

**Topics:**

- Input guardrails — validate what comes in

- Output guardrails — validate what goes out

- Tripwires — halt execution immediately on violation

- Parallel vs blocking execution modes

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from pydantic import BaseModel

# Notebook-specific imports
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrail,
    InputGuardrailTripwireTriggered,
    OutputGuardrail,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
    output_guardrail,
)

MODEL = "gpt-5-mini"


print("✅ Ready!")

---

## 🎯 The Problem

Agents respond to whatever they're asked — including off-topic requests, harmful prompts, or questions outside their scope.

Instructions encourage good behavior — guardrails enforce it, and they can't be overridden by the model.

Guardrails let you validate inputs and outputs so unsafe or off-topic content never reaches the user.

---

## 🛡️ Part 1: How Guardrails Work

Guardrails wrap the agent and intercept at two points:
```
User Input
↓
[Input Guardrail]  ← validates the input
↓ (if passes)
Agent runs
↓
[Output Guardrail] ← validates the output
↓ (if passes)
Response delivered
```

The diagram above shows the *logical* flow — what happens from a student's perspective.

By default, the input guardrail and agent run concurrently — the agent's result is discarded if the tripwire fires.

If a guardrail triggers its **tripwire** — the stop signal that fires when a violation is detected — execution halts and raises an exception instead of returning the agent's response.

Use blocking mode (Part 5) to guarantee the agent never starts if the guardrail fires.

---

## ✅ Part 2: Rule-Based Input Guardrail

The simplest guardrail — no LLM needed.

Just a function that checks the input against rules and returns a pass/fail verdict.

Use rule-based guardrails when the violation is easy to define in code — keywords, length limits, required formats, or simple allow/block lists.

Reach for LLM-based (Part 3) when the check needs judgment.

In [ ]:
# The SDK calls every guardrail with the same signature: ctx is the run context (used in Part 3 for tracing),
# agent is the agent being guarded, and input is the user message. You can ignore parameters you don't need.

# A simple keyword-based guardrail
@input_guardrail
async def topic_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    """Block requests unrelated to cooking."""
    input_text = input if isinstance(input, str) else str(input)

    # Off-topic keywords
    off_topic_terms = ["politics", "stocks", "weather", "sports", "news"]
    is_off_topic = any(term in input_text.lower() for term in off_topic_terms)

    return GuardrailFunctionOutput(
        tripwire_triggered=is_off_topic,
        # output_info is surfaced in tracing/logs — useful for debugging why a guardrail fired.
        # Can be a string or a Pydantic model (Part 4) when you want structured detail.
        output_info="Off-topic request detected" if is_off_topic else "OK"
    )


# The @input_guardrail decorator marks the function as a guardrail; wrapping it in InputGuardrail(...)
# is what attaches it to the agent and lets you configure parameters like run_in_parallel (Part 5).
cooking_agent = Agent(
    name="CookingAssistant",
    instructions="You are a cooking assistant. Help with recipes, techniques, and ingredients.",
    model=MODEL,
    input_guardrails=[InputGuardrail(guardrail_function=topic_guardrail)]
)

# --------------------------------------------------------------
print("✅ Cooking agent with input guardrail ready")

#### Test: On-Topic Request

In [ ]:
print("🧪 RULE-BASED GUARDRAIL TEST")
print("=" * 60)

# On-topic — passes
try:

    result = await Runner.run(cooking_agent, input="How do I make a good pasta sauce?")

    print("✅ On-topic request passed")
    print(f"Response: {result.final_output}")
except InputGuardrailTripwireTriggered:
    print("❌ Unexpectedly blocked")

#### Test: Off-Topic Request

In [ ]:
# Off-topic — blocked
try:

    result = await Runner.run(cooking_agent, input="What are today's top news stories?")

    print("Response:", result.final_output)
except InputGuardrailTripwireTriggered as e:
    print("🛑 Input blocked by guardrail")
    print(f"   Reason: {e}")

print("=" * 60)

---

## 🤖 Part 3: LLM-Based Input Guardrail

For nuanced checks that simple keyword matching can't handle, use a small, fast agent as the guardrail.

Pydantic `output_type` gives structured verdicts.

In [ ]:
class TopicCheck(BaseModel):
    is_cooking_related: bool
    reasoning: str


topic_checker_instructions = (
    "Determine if the user's message is related to cooking, recipes, food, or ingredients.\n"
    "Return is_cooking_related=True only if it's clearly about cooking or food."
)

topic_checker_agent = Agent(
    name="TopicChecker",
    instructions=topic_checker_instructions,
    model=MODEL,
    output_type=TopicCheck
)


@input_guardrail
async def llm_topic_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    """Use an LLM to check if input is cooking-related."""

    result = await Runner.run(topic_checker_agent, input, context=ctx.context)  # Pass the run context so the checker agent shares state and links into the same trace

    check = result.final_output
    return GuardrailFunctionOutput(
        tripwire_triggered=not check.is_cooking_related,
        output_info=check.reasoning
    )


smart_cooking_agent = Agent(
    name="SmartCookingAssistant",
    instructions="You are a cooking assistant. Help with recipes, techniques, and ingredients.",
    model=MODEL,
    input_guardrails=[InputGuardrail(guardrail_function=llm_topic_guardrail)]
)

# --------------------------------------------------------------
print("✅ Smart cooking agent with LLM guardrail ready")

#### Test: Ambiguous Request

In [ ]:
print("🤖 LLM GUARDRAIL TEST")
print("=" * 60)

test_inputs = [
    "What herbs pair well with chicken?",         # Should pass
    "Can you write me a Python script?",           # Should block
    "What wine goes with pasta?",                  # Should pass (food-adjacent)
]

for test_input in test_inputs:
    try:

        result = await Runner.run(smart_cooking_agent, input=test_input)

        print(f"✅ PASS: {test_input}")
        print(f"   → {result.final_output[:100]}")
    except InputGuardrailTripwireTriggered:
        print(f"🛑 BLOCKED: {test_input}")
    print()

print("=" * 60)

### 💡 Why This Works

The guardrail agent uses a small, fast model to classify intent.

It catches nuanced off-topic requests that keyword matching would miss — like a Python scripting question that contains no off-topic keywords.

---

## 📤 Part 4: Output Guardrail

Output guardrails validate the agent's response before it's delivered.

Useful for blocking responses that contain sensitive data, violate policy, or don't meet quality standards.

Like input guardrails, output guardrails can be rule-based (as shown here) or LLM-based — Exercise 2 walks through the LLM-based pattern.

This word-count example is deliberately simple — the same pattern (let the agent answer normally, then inspect the final output before it reaches the user) is how you'd block PII leaks, policy violations, or low-quality outputs in a real app.

In [ ]:
class LengthCheck(BaseModel):
    is_acceptable: bool
    word_count: int


@output_guardrail
async def length_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    output: str
) -> GuardrailFunctionOutput:
    """Block responses over 30 words — enforce concise answers."""
    word_count = len(output.split())
    is_too_long = word_count > 30

    return GuardrailFunctionOutput(
        tripwire_triggered=is_too_long,
        output_info=LengthCheck(
            is_acceptable=not is_too_long,
            word_count=word_count
        )
    )


concise_agent = Agent(
    name="ConciseAssistant",
    instructions="You are a helpful assistant. Answer questions clearly.",
    model=MODEL,
    output_guardrails=[OutputGuardrail(guardrail_function=length_guardrail)]
)

# --------------------------------------------------------------
print("✅ Concise agent with output guardrail ready (30 word limit)")

#### Test: Short vs Long Response

In [ ]:
print("📤 OUTPUT GUARDRAIL TEST")
print("=" * 60)

# Short answer — should pass
try:

    result = await Runner.run(concise_agent, input="What is 2 + 2?")

    print(f"✅ Short response passed ({len(result.final_output.split())} words)")
    print(f"   {result.final_output}")
except OutputGuardrailTripwireTriggered as e:
    print(f"🛑 Blocked: {e}")

print()

# Long response — should trigger the output guardrail
try:
    result2 = await Runner.run(
        concise_agent,
        input="Write at least 80 words explaining the history of pasta."
    )
    print(f"Short response passed: {result2.final_output}")
except OutputGuardrailTripwireTriggered:
    print("✅ Output guardrail fired — response exceeded 30 words")

print("=" * 60)

## ⚙️ Part 5: Blocking Mode

This isn't a new kind of guardrail — it changes *when* an input guardrail runs relative to the main agent.

Use blocking mode (`run_in_parallel=False`) when your guardrail is cheap (keyword or simple rule) but the main agent is expensive — the agent never starts if the input is invalid, saving both latency and tokens.

Stick with the default parallel mode when most requests are expected to pass and you want faster responses.

#### Test: Blocking Mode

In [ ]:
print("⚙️ BLOCKING MODE TEST")
print("=" * 60)

# Off-topic request — guardrail runs first, agent never starts
try:

    result = await Runner.run(blocking_agent, input="What's the latest news in sports?")

    print(f"Response: {result.final_output}")
except InputGuardrailTripwireTriggered:
    print("🛑 Blocked — guardrail ran first, agent never started")
    print("   No tokens spent on main agent")

print("=" * 60)

---

## 💪 Practice Exercises

### Exercise 1: Language Guardrail

*Covers: input guardrails — rule-based filtering*

Build an input guardrail that blocks non-English messages using a rule-based check.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Language Guardrail
# --------------------------------------------------------------
# Objective: Block messages that appear to be non-English.

# TODO 1: Create an @input_guardrail function that:
#           - Checks if the input contains only ASCII characters
#           - Triggers the tripwire if non-ASCII characters are found
#           (This is a simple heuristic — not perfect, but teaches the pattern)

# TODO 2: Create an Agent with this guardrail attached

# TODO 3: Test with:
#           - "Hello, how are you?" (should pass)
#           - "Bonjour, comment allez-vous?" (may pass — French uses ASCII)
#           - "こんにちは" (should block — Japanese uses non-ASCII)
#           - Print whether each passed or was blocked

# --- Write your code below this line ---

### Exercise 2: Sentiment Output Guardrail

*Covers: output guardrails — tone filtering*

Build an output guardrail that blocks responses with a negative or harsh tone.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Sentiment Output Guardrail
# --------------------------------------------------------------
# Objective: Block agent responses that are harsh or dismissive.

# TODO 1: Create a Pydantic model ToneCheck with:
#           - is_appropriate: bool
#           - reasoning: str

# TODO 2: Create a tone_checker_agent that classifies
#           whether a response is appropriate and professional
#           output_type=ToneCheck

# TODO 3: Create an @output_guardrail that runs the tone checker
#           and triggers the tripwire if is_appropriate=False

# TODO 4: Create an Agent with this output guardrail

# TODO 5: Test it — try to provoke a harsh response with a
#           rude input and see if the guardrail catches it

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Two Guardrail Types:**

- `input_guardrails` — validate user input at the start of a run; by default they run in parallel unless you set `run_in_parallel=False`

- `output_guardrails` — run on the agent's response before it's delivered
<br>
<br>

**Two Implementation Styles:**

- Rule-based — fast, no LLM, good for clear-cut checks (keywords, length, format)

- LLM-based — handles nuance, use a small fast model to keep costs low
<br>
<br>

**Tripwires Halt Execution:**

- `tripwire_triggered=True` raises `InputGuardrailTripwireTriggered` or `OutputGuardrailTripwireTriggered`

- When guardrails are attached, handle exceptions explicitly — wrap `Runner.run()` in try/except for runs that may trip them
<br>
<br>

**Parallel vs blocking mode:**

- Default (parallel): guardrail and agent start together — better latency, but agent may run before the guardrail result is known

- Blocking (`run_in_parallel=False`): guardrail runs first — no tokens spent on the main agent if blocked

- Use blocking mode when the guardrail check is cheap and the main agent call is expensive

---

## 📍 Next Step

**Notebook 23: Prompt Injection & Tool Safety**  

Learn how attackers exploit agents through injected instructions and how to build defenses into your tool design.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-22-guardrails)

---